# Feature Selection for Multi-band Light Curves (New Dataset)

Updated to work with **`light_curves.csv`** (columns: `oid, mjd, fid, mag/magpsf, e_mag/sigmapsf, ra, dec, isdiffpos/isdiff`).

**Goal:** choose a compact, robust set of input features for **self-supervised contrastive learning** on per-object light-curve sequences.

Because SSL has **no labels**, we avoid label-based feature selection and instead use:
- Missingness + basic sanity checks
- Low-variance filtering
- Correlation redundancy filtering
- (Optional) keep `ra/dec` out to avoid spatial leakage

Outputs:
- `selected_feature_names.json`
- optional `light_curves_sequences_selected.npz` (if you start from an NPZ sequences file)


In [11]:
import os, json
import numpy as np
import pandas as pd

DATA_CSV = os.environ.get('LIGHT_CURVES_CSV', 'light_curves.csv')
SEQ_NPZ  = os.environ.get('LIGHT_CURVES_SEQ_NPZ', 'outputs/light_curves_sequences.npz')
OUT_DIR  = 'outputs'
os.makedirs(OUT_DIR, exist_ok=True)

def load_light_curves_csv(path: str) -> pd.DataFrame:
    # works in notebook + local
    if os.path.exists(path):
        return pd.read_csv(path)
    sandbox_path = '/mnt/data/' + os.path.basename(path)
    if os.path.exists(sandbox_path):
        return pd.read_csv(sandbox_path)
    raise FileNotFoundError(f'Could not find {path} (or {sandbox_path})')

df = load_light_curves_csv(DATA_CSV)
print('Rows:', len(df), 'Cols:', list(df.columns))
df.head()

Rows: 20000 Cols: ['oid', 'mjd', 'fid', 'mag', 'e_mag', 'magpsf', 'sigmapsf', 'ra', 'dec', 'isdiffpos']


,oid,mjd,fid,mag,e_mag,magpsf,sigmapsf,ra,dec,isdiffpos
0,ZTF18aazeojq,58278.407130,1,NaN,NaN,16.692076,0.025775,307.792636,51.134943,-1
1,ZTF18aazeojq,58281.403681,1,NaN,NaN,16.631727,0.024336,307.792558,51.134826,-1
2,ZTF18aazeojq,58285.413102,2,NaN,NaN,16.383300,0.027792,307.792625,51.134461,1
3,ZTF18aazeojq,58287.404167,1,NaN,NaN,16.658768,0.025382,307.792841,51.134746,-1
4,ZTF18aazeojq,58288.407836,1,NaN,NaN,16.912527,0.146301,307.792625,51.135020,1


In [12]:
# ---- Standardize columns to a consistent schema ----
df = df.copy()

# magnitude
if 'magpsf' in df.columns:
    df['mag_used'] = df['magpsf']
elif 'mag' in df.columns:
    df['mag_used'] = df['mag']
else:
    raise ValueError('No magnitude column found (expected magpsf or mag).')

# magnitude error
if 'sigmapsf' in df.columns:
    df['err_used'] = df['sigmapsf']
elif 'e_mag' in df.columns:
    df['err_used'] = df['e_mag']
else:
    # allow missing, but warn
    df['err_used'] = np.nan
    print('Warning: no error column found (expected sigmapsf or e_mag).')

# diff flag
diff_col = None
for c in ['isdiffpos','isdiff']:
    if c in df.columns:
        diff_col = c
        break
if diff_col is None:
    df['isdiff_flag'] = 0
else:
    # normalize to 0/1
    x = df[diff_col]
    if x.dtype == object:
        df['isdiff_flag'] = x.astype(str).str.lower().isin(['t','true','1','y','yes','pos','p']).astype(int)
    else:
        df['isdiff_flag'] = (x.fillna(0).astype(float) != 0).astype(int)

# band / filter
if 'fid' not in df.columns:
    raise ValueError('Missing fid column (filter id).')

df = df.dropna(subset=['oid','mjd','fid','mag_used'])
df = df.sort_values(['oid','mjd']).reset_index(drop=True)

print('After basic cleaning:', df.shape)
df[['oid','mjd','fid','mag_used','err_used','isdiff_flag']].head()

After basic cleaning: (20000, 13)


,oid,mjd,fid,mag_used,err_used,isdiff_flag
0,ZTF18aajtltx,58252.436134,1,19.838300,0.104995,1
1,ZTF18aajtltx,58252.443218,1,19.928700,0.133612,1
2,ZTF18aajtltx,58252.491157,2,19.125100,0.137872,1
3,ZTF18aajtltx,58261.453738,1,20.198800,0.143248,1
4,ZTF18aajtltx,58272.391377,1,19.766142,0.178495,1


## Feature Engineering Candidates

We build per-observation features that are commonly useful for time-series encoders:
- `mag_used`
- `err_used` (filled with median if missing)
- `dt` = time delta between consecutive observations (per object)
- `fid` one-hot (`fid_g`, `fid_r`, `fid_i` if values are 1/2/3)
- `isdiff_flag`

Optionally:
- `ra`, `dec` can be included as **static object-level** features, but by default we exclude them to avoid spatial leakage.


In [13]:
INCLUDE_RADEC = False  # set True only if you *really* want spatial features

# dt per object
df['dt'] = df.groupby('oid')['mjd'].diff().fillna(0.0)

# fill err
if df['err_used'].isna().all():
    df['err_used'] = 0.0
else:
    df['err_used'] = df['err_used'].fillna(df['err_used'].median())

# fid one-hot (robust to unexpected fid values)
fid_vals = sorted(df['fid'].dropna().unique().tolist())
for f in fid_vals:
    df[f'fid_{int(f)}'] = (df['fid'].astype(int) == int(f)).astype(int)

feature_cols = ['mag_used','err_used','dt','isdiff_flag'] + [f'fid_{int(f)}' for f in fid_vals]
if INCLUDE_RADEC and ('ra' in df.columns) and ('dec' in df.columns):
    feature_cols += ['ra','dec']

print('Candidate feature columns:', feature_cols)
df[feature_cols].describe(include='all')

Candidate feature columns: ['mag_used', 'err_used', 'dt', 'isdiff_flag', 'fid_1', 'fid_2']


,mag_used,err_used,dt,isdiff_flag,fid_1,fid_2
count,20000.000000,20000.000000,20000.000000,20000.0,20000.000000,20000.000000
mean,16.593490,0.082575,0.680289,1.0,0.526100,0.473900
std,1.177505,0.045597,4.363703,0.0,0.499331,0.499331
min,13.544434,0.009013,0.000000,1.0,0.000000,0.000000
25%,15.501700,0.055472,0.000475,1.0,0.000000,0.000000
50%,16.554196,0.077195,0.017263,1.0,1.000000,0.000000
75%,17.567647,0.103613,0.233192,1.0,1.000000,1.000000
max,20.537700,0.499309,257.128866,1.0,1.000000,1.000000


## Unsupervised Feature Screening

### 1) Drop near-constant features (low variance)
### 2) Drop highly correlated numeric features (redundant)

Notes:
- One-hot `fid_*` columns are kept even if correlated (they’re categorical indicators).
- Correlation filtering is applied only to continuous features.


In [14]:
from sklearn.feature_selection import VarianceThreshold

X = df[feature_cols].astype(float)

# Separate continuous vs categorical-ish (one-hot)
one_hot_cols = [c for c in feature_cols if c.startswith('fid_')]
cont_cols = [c for c in feature_cols if c not in one_hot_cols]

# 1) low variance filter on continuous columns
vt = VarianceThreshold(threshold=1e-6)
X_cont = X[cont_cols]
vt.fit(X_cont)
kept_cont = [c for c, keep in zip(cont_cols, vt.get_support()) if keep]

# 2) correlation redundancy filter on kept continuous columns
corr = X[kept_cont].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.98)]
kept_cont2 = [c for c in kept_cont if c not in to_drop]

selected_features = kept_cont2 + one_hot_cols
print('Kept continuous:', kept_cont2)
print('Dropped (high corr):', to_drop)
print('Selected feature set:', selected_features)

# Save
out_json = os.path.join(OUT_DIR, 'selected_feature_names.json')
with open(out_json, 'w') as f:
    json.dump({'selected_features': selected_features, 'fid_values': fid_vals, 'include_radec': INCLUDE_RADEC}, f, indent=2)
print('Saved:', out_json)

Kept continuous: ['mag_used', 'err_used', 'dt']
Dropped (high corr): []
Selected feature set: ['mag_used', 'err_used', 'dt', 'fid_1', 'fid_2']
Saved: outputs\selected_feature_names.json


## Optional: Apply Feature Selection to Sequence NPZ

If your preprocessing notebook produced a fixed-length sequence NPZ (e.g., `outputs/light_curves_sequences.npz`),
this cell will create a new NPZ containing only the selected features.

Expected NPZ keys (common pattern):
- `X` shape `(N, T, F)`
- `oids` shape `(N,)`
- `feature_names` shape `(F,)`


In [15]:
import glob

def find_sequence_npz(default_path: str):
    candidates = []
    if default_path:
        candidates.append(default_path)
    # search outputs/ for any npz files as fallback
    candidates += sorted(glob.glob(os.path.join(OUT_DIR, '*.npz')))
    seen = set()
    for p in candidates:
        if p in seen:
            continue
        seen.add(p)
        sandbox_path = '/mnt/data/' + os.path.basename(p)
        for candidate in (p, sandbox_path):
            if not os.path.exists(candidate):
                continue
            try:
                npz = np.load(candidate, allow_pickle=True)
            except Exception:
                continue
            keys = list(npz.keys())
            if 'X' in keys and 'feature_names' in keys:
                return npz, candidate, keys
    return None, None, []

npz, used_path, keys = find_sequence_npz(SEQ_NPZ)
if npz is None:
    candidates = [SEQ_NPZ] + sorted(glob.glob(os.path.join(OUT_DIR, '*.npz')))
    print('No sequence NPZ found. Skipping. Searched candidates:', candidates)
else:
    print('Using NPZ:', used_path, 'keys:', keys)
    Xseq = npz['X']
    fn = [str(x) for x in npz['feature_names'].tolist()]
    keep_idx = [i for i, name in enumerate(fn) if name in selected_features]
    if len(keep_idx) == 0:
        print('No selected features found in NPZ feature_names. Available:', fn)
    else:
        Xsel = Xseq[:, :, keep_idx]
        fn_sel = [fn[i] for i in keep_idx]

        out_npz = os.path.join(OUT_DIR, 'light_curves_sequences_selected.npz')
        save_dict = {'X': Xsel, 'feature_names': np.array(fn_sel, dtype=object)}
        if 'oids' in keys:
            save_dict['oids'] = npz['oids']
        np.savez_compressed(out_npz, **save_dict)
        print('Saved:', out_npz, 'X:', Xsel.shape, 'features:', fn_sel)

No sequence NPZ found. Skipping. Searched candidates: ['outputs/light_curves_sequences.npz', 'outputs\\light_curve_pairs.npz']
